# Aufgaben 6 – Modell 1: K-Nearest Neighbors (KNN)
**Kurs:** NKDS – Machine Learning | **Dozentin:** Prof. Dr. Jennifer Schoch, DHBW Karlsruhe  
**Datensatz:** LLM - Detect AI Generated Text | **Gruppe:** [Namen eintragen] | **Datum:** [Datum]

---

## Theoretischer Hintergrund

**K-Nearest Neighbors (KNN)** ist ein instanzbasierter Lernalgorithmus ohne explizite Trainingsphase:
- Zur Klassifikation eines neuen Datenpunkts werden die **k nächsten Nachbarn** im Feature-Raum gesucht.
- Die Klasse wird per **Mehrheitsvotum** der k Nachbarn bestimmt.
- Die Distanz wird typischerweise mit der **euklidischen Distanz** gemessen.
- **Hyperparameter k**: Klein → hohes Overfitting-Risiko; Groß → Glattere Entscheidungsgrenze, aber ggf. Underfitting.

> ⚠️ **Hinweis für diesen Datensatz:** KNN auf rohem TF-IDF ist sehr rechenintensiv (10.000-dimensionaler Raum). Wir reduzieren die Dimensionen mit **TruncatedSVD (LSA)** und begrenzen die Datenmenge für schnelle Ausführung.


## 0. Setup & Daten laden

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)
from sklearn.pipeline import Pipeline

df = pd.read_csv("train_essays.csv").dropna(subset=["text"]).drop_duplicates(subset=["text"])
print(f"Datensatz: {df.shape[0]} Zeilen, {df.shape[1]} Spalten")
print(f"Klassenverteilung:\n{df['generated'].value_counts()}")


## 1a. Train / Validation / Test Split

In [ ]:
X = df["text"]
y = df["generated"]

X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.25, random_state=42, stratify=y_trainval
)

print(f"Trainingsdaten:    {len(X_train):>5} Samples ({len(X_train)/len(X)*100:.0f}%)")
print(f"Validierungsdaten: {len(X_val):>5} Samples ({len(X_val)/len(X)*100:.0f}%)")
print(f"Testdaten:         {len(X_test):>5} Samples ({len(X_test)/len(X)*100:.0f}%)")


## 1b. Feature Extraction – TF-IDF + Dimensionsreduktion (LSA)

KNN berechnet Abstände zwischen allen Punkten – im 10.000-dimensionalen TF-IDF-Raum ist das sehr langsam.  
Lösung: **Truncated SVD** (= Latent Semantic Analysis) reduziert die Dimensionen auf 100, ohne wesentliche Information zu verlieren.


In [ ]:
# TF-IDF → Dimensionsreduktion auf 100 Komponenten → Normalisierung
tfidf = TfidfVectorizer(max_features=5000, sublinear_tf=True)
svd   = TruncatedSVD(n_components=100, random_state=42)
norm  = Normalizer(copy=False)

# Fit nur auf Trainingsdaten!
X_train_tfidf = tfidf.fit_transform(X_train)
X_train_svd   = svd.fit_transform(X_train_tfidf)
X_train_norm  = norm.fit_transform(X_train_svd)

X_val_norm    = norm.transform(svd.transform(tfidf.transform(X_val)))
X_test_norm   = norm.transform(svd.transform(tfidf.transform(X_test)))

print(f"Shape nach TF-IDF:  {X_train_tfidf.shape}")
print(f"Shape nach SVD:     {X_train_svd.shape}")
print(f"Erklärte Varianz durch SVD: {svd.explained_variance_ratio_.sum()*100:.1f}%")


## 1b–1c. Modell trainieren & Hyperparameter k vergleichen

Wir testen verschiedene Werte für k, um den optimalen Parameter zu finden.


In [ ]:
k_values = [1, 3, 5, 7, 10, 15, 20, 30]
results = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k, metric="euclidean", n_jobs=-1)
    knn.fit(X_train_norm, y_train)

    train_acc = accuracy_score(y_train, knn.predict(X_train_norm))
    val_acc   = accuracy_score(y_val,   knn.predict(X_val_norm))

    results.append({"k": k, "Train Accuracy": round(train_acc, 4), "Val Accuracy": round(val_acc, 4)})
    print(f"k={k:>2} | Train: {train_acc:.4f} | Val: {val_acc:.4f}")

results_df = pd.DataFrame(results)
results_df


## 1f. Visualisierung – Einfluss von k auf die Accuracy

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

ax.plot(results_df["k"], results_df["Train Accuracy"], marker="o", label="Train Accuracy", color="steelblue")
ax.plot(results_df["k"], results_df["Val Accuracy"],   marker="s", label="Val Accuracy",   color="tomato", linestyle="--")

best_k = results_df.loc[results_df["Val Accuracy"].idxmax(), "k"]
ax.axvline(x=best_k, color="gray", linestyle=":", label=f"Bestes k = {best_k}")

ax.set_xlabel("k (Anzahl Nachbarn)")
ax.set_ylabel("Accuracy")
ax.set_title("KNN: Einfluss von k auf Train- vs. Validierungs-Accuracy")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nBestes k (nach Val Accuracy): k = {best_k}")


**Beobachtung:**  
- Bei **kleinem k** (z.B. k=1) ist der Train-Accuracy sehr hoch → starkes Overfitting.  
- Mit wachsendem k sinkt der Train-Val-Gap → das Modell generalisiert besser.  
- Ab einem bestimmten k sinkt auch die Val-Accuracy wieder → Underfitting.  

[Eigene Beobachtungen zu Ihrem k-Verlauf eintragen]


## 1d. Bestes Modell auf Testdaten evaluieren

In [ ]:
# Bestes k auswählen
best_k = results_df.loc[results_df["Val Accuracy"].idxmax(), "k"]
print(f"Gewähltes k: {best_k}")

# Finales Modell trainieren und auf Testdaten anwenden
knn_best = KNeighborsClassifier(n_neighbors=best_k, metric="euclidean", n_jobs=-1)
knn_best.fit(X_train_norm, y_train)

y_pred_test = knn_best.predict(X_test_norm)
test_acc    = accuracy_score(y_test, y_pred_test)

print(f"Test Accuracy: {test_acc:.4f}")
print("\nKlassifikationsbericht:")
print(classification_report(y_test, y_pred_test, target_names=["Mensch (0)", "KI (1)"]))


In [ ]:
# Konfusionsmatrix
cm = confusion_matrix(y_test, y_pred_test)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Mensch (0)", "KI (1)"])
disp.plot(cmap="Blues")
plt.title(f"Konfusionsmatrix – KNN (k={best_k})")
plt.tight_layout()
plt.show()


## 1e. Stichprobe: Vorhersage vs. echtem Wert

In [ ]:
sample_idx   = X_test.sample(10, random_state=7).index
sample_texts = X_test.loc[sample_idx]
sample_true  = y_test.loc[sample_idx]
sample_norm  = norm.transform(svd.transform(tfidf.transform(sample_texts)))
sample_pred  = knn_best.predict(sample_norm)

pd.DataFrame({
    "Text (Ausschnitt)": [t[:80] + "..." for t in sample_texts],
    "Wahrer Wert":       sample_true.values,
    "Vorhersage":        sample_pred,
    "Korrekt?":          ["✅" if t==p else "❌" for t, p in zip(sample_true.values, sample_pred)]
})


## 1g. Erkenntnisse & Bewertung

### Zusammenfassung

| Bestes k | Val Accuracy | Test Accuracy |
|----------|-------------|---------------|
| [eintragen] | [eintragen] | [eintragen] |

### Stärken von KNN
- Kein explizites Training nötig (lazy learner)
- Einfach zu verstehen und zu implementieren
- Nicht-lineare Entscheidungsgrenzen möglich

### Schwächen von KNN
- Sehr langsam bei großen Datensätzen (muss alle Distanzen berechnen)
- Sensitiv gegenüber irrelevanten Features und hoher Dimensionalität (Fluch der Dimensionalität)
- Dimensionsreduktion notwendig für Textdaten

### Was haben wir erwartet / Was hat überrascht?
[Eigene Reflexion der Gruppe eintragen]
